In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CanineModel, CanineTokenizer
import pandas as pd
from classes.style_encoder import StyleEncoder
from classes.conversational_dataset import ConversationDataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import string
import emoji

device = "cuda" if torch.cuda.is_available() else "cpu"

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1226 17:24:57.857000 31252 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
def author_contrastive_loss(z, authors, temperature=0.1):
    """
    z: [B, D] normalized embeddings
    authors: [B] author ids
    """
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temperature  # [B, B]

    # mask self similarity
    mask = torch.eye(sim.size(0), device=sim.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: same author
    pos_mask = authors.unsqueeze(0) == authors.unsqueeze(1)
    pos_mask = pos_mask & ~mask  # remove self

    # log-softmax over rows
    log_prob = F.log_softmax(sim, dim=1)

    # average over positives
    loss = -log_prob[pos_mask].mean()
    return loss

def compute_style_score(texts):
    scores = []
    for t in texts:
        if not isinstance(t, str):
            scores.append(0.0)
            continue

        emoji_count = sum(c in emoji.EMOJI_DATA for c in t)
        caps_ratio = sum(c.isupper() for c in t) / max(len(t), 1)
        punct_count = sum(c in string.punctuation for c in t)

        # simple scalar style intensity
        score = emoji_count + caps_ratio * 5 + punct_count * 0.5
        scores.append(score)

    return torch.tensor(scores, dtype=torch.float32)

def style_contrastive_loss(z_anchor, z_pos, z_neg, temperature=0.1):
    z_anchor = F.normalize(z_anchor, dim=1)
    z_pos = F.normalize(z_pos, dim=1)
    z_neg = F.normalize(z_neg, dim=1)

    pos_sim = torch.sum(z_anchor * z_pos, dim=1) / temperature
    neg_sim = torch.sum(z_anchor * z_neg, dim=1) / temperature

    logits = torch.stack([pos_sim, neg_sim], dim=1)
    labels = torch.zeros(z_anchor.size(0), dtype=torch.long, device=z_anchor.device)

    return F.cross_entropy(logits, labels)

def style_metric_loss(z_i, z_j, s_i, s_j):
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)

    z_dist = 1 - torch.sum(z_i * z_j, dim=1)
    style_dist = torch.abs(s_i - s_j)

    return F.mse_loss(z_dist, style_dist)

In [3]:
from collections import defaultdict
from torch.utils.data import Sampler
import random

def build_author_index_from_rows(rows, author2id):
    author_to_indices = defaultdict(list)
    for idx, row in enumerate(rows):
        author_id = author2id[row["Author"]]
        author_to_indices[author_id].append(idx)
    return author_to_indices


class AuthorBalancedBatchSampler(Sampler):
    def __init__(
        self,
        dataset,
        authors_per_batch=8,
        samples_per_author=4
    ):
        self.dataset = dataset
        self.authors_per_batch = authors_per_batch
        self.samples_per_author = samples_per_author

        self.author_to_indices = build_author_index_from_rows(
            dataset.rows,
            dataset.author2id
        )

        self.authors = list(self.author_to_indices.keys())
        self.batch_size = authors_per_batch * samples_per_author

    def __iter__(self):
        random.shuffle(self.authors)

        for i in range(0, len(self.authors), self.authors_per_batch):
            batch_authors = self.authors[i:i + self.authors_per_batch]
            batch_indices = []

            for author in batch_authors:
                indices = self.author_to_indices[author]
                if len(indices) >= self.samples_per_author:
                    chosen = random.sample(indices, self.samples_per_author)
                else:
                    chosen = random.choices(indices, k=self.samples_per_author)
                batch_indices.extend(chosen)

            yield batch_indices

    def __len__(self):
        return len(self.authors) // self.authors_per_batch
    

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "author": torch.tensor([b["author"] for b in batch]),
        "content": [b["content"] for b in batch]
    }

In [4]:
rows = pd.read_csv("../data/train/retriever_train.csv")
rows['Content'] = rows['Content'].astype("string")
rows = rows.to_dict(orient="records")

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
dataset = ConversationDataset(rows, tokenizer)

sampler = AuthorBalancedBatchSampler(
    dataset=dataset,
    authors_per_batch=8,
    samples_per_author=4
)


loader = DataLoader(
    dataset,
    batch_sampler=sampler,
    collate_fn=collate_fn,
)


In [5]:
from collections import Counter

def preview_batches(loader, num_batches=3):
    for i, batch in enumerate(loader):
        authors = batch["author"].tolist()
        counts = Counter(authors)

        print(f"\nBatch {i+1}")
        print("Batch size:", len(authors))
        print("Unique authors:", len(counts))
        print("Samples per author:", dict(counts))

        if i + 1 >= num_batches:
            break

preview_batches(loader, num_batches=2)


Batch 1
Batch size: 32
Unique authors: 8
Samples per author: {125: 4, 214: 4, 223: 4, 134: 4, 179: 4, 228: 4, 220: 4, 233: 4}

Batch 2
Batch size: 32
Unique authors: 8
Samples per author: {197: 4, 143: 4, 201: 4, 139: 4, 165: 4, 154: 4, 187: 4, 106: 4}


In [ ]:
model = StyleEncoder().to(device)

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.proj.parameters(), "lr": 5e-4}
])

epochs = 500 
temperature = 0.1
lambda_metric = 0.5 # style contrastive only

model.train()

for epoch in range(epochs):
    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        authors = batch["author"].to(device)

        z = model(input_ids, attention_mask)

        loss = author_contrastive_loss(
            z=z,
            authors=authors,
            temperature=temperature
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

save_dir = "../models/style_retriever_author_contrastive"

model.eval()
model.cpu()  # move to CPU before saving

model.encoder.save_pretrained(save_dir)
torch.save(model.proj.state_dict(), f"{save_dir}/projection_head.pt")
tokenizer.save_pretrained(save_dir)


Epoch 1: 31it [00:12,  2.44it/s]                        


Epoch 1 | Loss: 3.5618


Epoch 2: 31it [00:11,  2.58it/s]                        


Epoch 2 | Loss: 3.4813


Epoch 3: 31it [00:12,  2.58it/s]                        


...

Epoch 499: 31it [00:11,  2.70it/s]                        


Epoch 499 | Loss: 3.2271


Epoch 500: 31it [00:11,  2.68it/s]                        


Epoch 500 | Loss: 3.2304


('../models/style_retriever_author_contrastive\\tokenizer_config.json',
 '../models/style_retriever_author_contrastive\\special_tokens_map.json',
 '../models/style_retriever_author_contrastive\\added_tokens.json')

In [7]:
# model = StyleEncoder().to(device)

# optimizer = torch.optim.AdamW([
#     {"params": model.encoder.parameters(), "lr": 1e-5},
#     {"params": model.proj.parameters(), "lr": 5e-4}
# ])

# epochs = 5
# temperature = 0.1
# lambda_metric = 0.5

# model.train()

# for epoch in range(epochs):
#     total_loss = 0

#     for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)

#         # raw text is required for style score
#         texts = batch["content"]


#         z = model(input_ids, attention_mask)

#         # ----- build positives / negatives -----
#         perm = torch.randperm(z.size(0))
#         z_pos = z[perm]

#         z_neg = torch.roll(z, shifts=1, dims=0)

#         # ----- style scores -----
#         style_scores = compute_style_score(texts).to(device)
#         s_i = style_scores
#         s_j = style_scores[perm]

#         # ----- losses -----
#         loss_contrastive = style_contrastive_loss(
#             z, z_pos, z_neg, temperature
#         )

#         loss_metric = style_metric_loss(
#             z, z_pos, s_i, s_j
#         )

#         loss = loss_contrastive + lambda_metric * loss_metric
        
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(loader)
#     print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

# # -----------------------
# # Save model
# # -----------------------
# save_dir = "../models/style_retriever_style_contrastive"
# model.eval()
# model.cpu()

# model.encoder.save_pretrained(save_dir)
# torch.save(model.proj.state_dict(), f"{save_dir}/projection_head.pt")
# tokenizer.save_pretrained(save_dir)